# N129 · 组合计数、容斥与Catalan

**目标：** 先确定计数对象和是否有序再写公式。

**先修：** N128, N064, N098。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 排列/组合；Pascal；乘加原理；容斥；Catalan；重复对象；模组合。

**配套讲解来源：** [同名逐章意见](../../comment/129_combinatorics_counting.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章要解决的问题是：**"有多少种选法/走法/构造法？"这类数数题，怎么算才能又准又快。** “学习目标”部分 说"先确定计数对象和是否有序再写公式"——这是数数前必答的两个问题：你数的东西是什么？选出的一组有没有先后顺序？答案不同，公式完全不同。

本章实现三个计数器：组合数 C(n,k)（从 n 个里选 k 个，不排序）、Catalan 数（一类"任何前缀都不能越界"的结构总数，比如合法括号串）、满射计数（把 n 个球塞进 k 个盒子且每个盒子都非空的方案数）。

给你具体到数字的例子：

- 输入：`n = 5, k = 2`（从 5 个人里选 2 个组队，不区分谁是第一个选中的）
- 输出：`binomial(5, 2) = 10`
- 为什么是 10？因为如果讲顺序，5×4 = 20 种；但"先甲后乙"和"先乙后甲"是同一队，每种搭配被数了 2 次，所以 20 ÷ 2 = 10。

再看满射的例子：输入 `n = 4, k = 3`（4 个球放 3 个盒子，盒子都不能空），输出 36。为什么不是简单的 3⁴ = 81？因为 81 里混着"有盒子空着"的放法，得把它们剔除——这正是 “运行示例”部分 表格演示的容斥过程。

## 2. 基础知识：先读懂这些词

- **排列 vs 组合**：选出来之后**在乎顺序**的是排列（P(n,k) = n!/(n−k)!，比如排队领奖的金银铜）；**不在乎顺序**的是组合（C(n,k) = n!/(k!(n−k)!)，比如组队）。C(n,k) 就是把 P(n,k) 除以"k 个元素自身的 k! 种排法"。
- **乘法原理 / 加法原理**：一件事分几步完成，总数是各步方案数的乘积（3 件上衣 × 2 条裤子 = 6 套穿搭）；几类互不重叠的方案二选一，总数是各类之和。所有计数公式 ultimately 都由这两条搭出来。
- **Pascal 三角形（杨辉三角）**：把 C(n,k) 摆成三角表，每格等于肩上两格之和（C(n,k) = C(n−1,k−1) + C(n−1,k)）。它既是记忆工具，也是打表计算组合数的递推法。
- **逐项乘除（保持整数精确）**：算 C(n,k) 不用 n!/(k!(n−k)!) 一次除完（分子分母都是天文数字），而是循环里做 k 次"乘一个、整除一个"，每一步都恰好整除、结果都是整数，全程不碰浮点数（浮点大数会丢精度）。
- **容斥原理（inclusion–exclusion）**：要数"全都覆盖到"的方案，可以先数全部，减掉"漏了某个"的，加上"被减多了的"，交替加减直到数准。就像统计参加两个社团的人数：|A|+|B| 把两边都参加的人算了两次，得减去 |A∩B| 一次。
- **Catalan 数**：一列神奇的计数数字 1, 1, 2, 5, 14, 42…，数的是"n 对括号的合法排列""n 个节点的二叉搜索树形态""不穿过对角线的网格路径"等一大类结构。公式 Cat(n) = C(2n,n)/(n+1)。
- **反射法**：推导 Catalan 公式的图办法：所有路径 C(2n,n) 条，其中非法路径（某步之后"右括号多于左括号"，即路径越过边界）和另一批路径一一对应，那批恰好 C(2n,n+1) 条，相减即得。
- **满射（onto function）**：值域里每个元素都被映射到的函数。"n 个球塞 k 个盒且无空盒"就是满射的直观形象。
- **模组合（本章知识点的延伸）**：当题目要求答案 mod 大质数时组合数要配模逆元；但本章刻意走纯整数路线（“正确性与复杂度”部分 说不引入模逆元前提），先把整数世界的公式算准。

## 3. 思路推导：从小例子开始

- **Step 1：先回答"有序吗"。** 同样"从 5 个里挑 2 个"，排队是 20 种，组队是 10 种。动笔算数之前先把这个问题钉死。
- **Step 2：组合数用逐项乘除算，保持精确。** C(n,k) = ((n−k+1) × (n−k+2) × … × n) ÷ (1 × 2 × … × k)。一边乘一边除，第 i 步的中间结果恰是 C(n−k+i, i)，永远整数。
- **Step 3：利用对称性省一半循环。** C(n,k) = C(n,n−k)，所以 k 大于一半时换用 n−k，循环次数取小者。
- **Step 4：Catalan 套组合公式。** Cat(n) = C(2n,n) − C(2n,n+1)，代数化简后等于 C(2n,n) ÷ (n+1)，除法必然整除。
- **Step 5：满射计数上容斥。** 全部函数 kⁿ 种；减去"漏掉某个指定目标"的，加上"漏掉某两个"的……交替加减 i 从 0 到 k，每项是 C(k,i)·(k−i)ⁿ。

**手算演示一：binomial(5, 2)。** k = min(2, 3) = 2，answer 从 1 开始：

| 轮次 i | 乘 (n−k+i) | 除 i | answer |
|--------|-----------|------|--------|
| 1 | ×4 | ÷1 | 4 |
| 2 | ×5 | ÷2 | 10 |

两轮得 10，和"20 ÷ 2 = 10"一致。注意每步整除不是巧合：第 i 轮结束时的 answer = C(n−k+i, i)，组合数当然是整数。

**手算演示二：catalan(3)。** C(2×3, 3) = C(6,3) = 20，再 ÷ (3+1) = 5。这 5 个正是 3 对括号的全部合法串：((()))、(()())、(())()、()(())、()()()。

**手算演示三：count_onto_small(4, 3) 的容斥过程**（“运行示例”部分 的表格就是这个 n=4、k=3 的实例，最终打印"满射数 36"）：

| 遗漏目标数 i | 选遗漏集合 C(3,i) | 剩余函数数 (3−i)⁴ | 带符号贡献 |
|-------------|------------------|------------------|-----------|
| 0 | 1 | 81 | +81 |
| 1 | 3 | 16 | −48 |
| 2 | 3 | 1 | +3 |
| 3 | 1 | 0 | −0 |

含义逐行读：i=0 是"所有 3⁴=81 个函数"；i=1 减去"指定某个盒子空着"的 2⁴=16 种、3 种指定法共减 48；i=2 加回"两个盒子同时空"的（刚才被减了两遍）1⁴=1 种、3 种选法共加 3；i=3 是"全空"，0⁴=0 没有贡献。总计 81 − 48 + 3 = 36。直觉核对：满射情形 4 球 3 盒的分布必是 2+1+1，选"拿两个球的盒子"3 种、再把 4 个球排列进去 C(4,2)×2 = 12 种，3 × 12 = 36，吻合。

**手算演示四：杨辉三角的加法路线（知识点 Pascal）。** 同样是 C(5,2)=10，不用乘除也能得：把第 n 行摆出来，每格等于肩上两格之和——

| 行号 | 内容 |
|------|------|
| 0 | 1 |
| 1 | 1 1 |
| 2 | 1 2 1 |
| 3 | 1 3 3 1 |
| 4 | 1 4 6 4 1 |
| 5 | 1 **5 10** 10 5 1 |

第 5 行第 2 格 = 4+6 = 10（加粗那对肩膀）。这条"只用加法"的递推 C(n,k)=C(n−1,k−1)+C(n−1,k) 是打表利器，也解释了为什么 C(n,k) 总是整数——它是从 1 开始一路加出来的。

## 4. 核心代码：binomial

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def binomial(n, k):
    if n < 0:
        raise ValueError('nonnegative n required')
    if k < 0 or k > n:
        return 0
    k = min(k, n - k)
    answer = 1
    for i in range(1, k + 1):
        answer = answer * (n - k + i) // i
    return answer
```

### 逐段读懂代码

“分段实现”部分 有三个函数。

**函数一：`binomial(n, k)` —— 组合数 C(n,k)。**

```python
def binomial(n, k):
    if n < 0:
        raise ValueError('nonnegative n required')
    if k < 0 or k > n:
        return 0
    k = min(k, n - k)
    answer = 1
    for i in range(1, k + 1):
        answer = answer * (n - k + i) // i
    return answer
```

- `if n<0: raise ValueError(...)`：挡住负的 n。组合数的定义域是 n ≥ 0，传负数多半是调用方算错了，报错比返回猜测值好。
- `if k<0 or k>n: return 0`：数学约定——从 n 个里选不出负数个，也选不出比 n 还多的，这两种情况方案数就是 0（不是报错，因为"0 种选法"是合法答案，比如 `binomial(3,5)==0` 这条测试）。
- `k=min(k,n-k)`：对称性 C(n,k) = C(n,n−k)。比如 C(100,98) 换成 C(100,2)，循环从 98 轮缩到 2 轮。
- `for i in range(1,k+1): answer=answer*(n-k+i)//i`：核心一行。每轮乘上分子下一项 (n−k+i) 再整除 i。**先乘后除的顺序很重要**：如果先 `answer//i` 可能除不尽丢精度；先乘再除，由于乘积含因子 i，整除必然精确。Python 大整数不会溢出，这条路稳。
- 整个函数没有任何浮点运算，“正确性与复杂度”部分 强调的"整数域计算、不引入模逆元前提"就落实在这里。

**函数二：`catalan(n)` —— Catalan 数 Cat(n)。**

```python
def catalan(n):
    if n < 0:
        raise ValueError('nonnegative n required')
    return binomial(2 * n, n) // (n + 1)
```

- `if n<0: raise ValueError(...)`：同样挡负数，n 是结构规模（括号对数），不能为负。
- `return binomial(2*n,n)//(n+1)`：直接套公式 Cat(n) = C(2n,n) ÷ (n+1)。注意用 `//` 整除——这不是碰运气，反射法保证 C(2n,n) − C(2n,n+1) = C(2n,n)/(n+1) 是精确整数，除法必然除尽。n=3 时 20//4 = 5，见第三节手算。

**函数三：`count_onto_small(n, k)` —— 满射个数（小规模版）。**

```python
def count_onto_small(n, k):
    if min(n, k) < 0:
        raise ValueError('nonnegative sizes required')
    if k > n:
        return 0
    return sum(((-1) ** i * binomial(k, i) * (k - i) ** n for i in range(k + 1)))
```

- `if min(n,k)<0: raise ValueError(...)`：`min(n,k)<0` 一石二鸟，n 或 k 任何一个为负都拦下。两参数同类校验时这种写法比两个 if 紧凑。
- `if k>n: return 0`：鸽子比鸽子笼还少时必然有空盒（抽屉原理），满射不存在，方案数 0。
- `return sum(...)`：容斥主公式 ∑ᵢ (−1)ⁱ · C(k,i) · (k−i)ⁿ。生成器从 i=0 累加到 k：`(k-i)**n` 是"排除 i 个指定目标后，n 个元素各自只剩 k−i 个去处"的函数总数；`(−1)**i` 实现交替加减；`binomial(k,i)` 数"选哪 i 个目标被遗漏"。三项相乘再求和，就是第三节表格那四行。

## 5. 分段实现：按顺序运行

**本章接口：** `binomial(n, k)`、`catalan(n)`、`count_onto_small(n, k)`

### binomial

In [1]:
def binomial(n, k):
    if n < 0:
        raise ValueError('nonnegative n required')
    if k < 0 or k > n:
        return 0
    k = min(k, n - k)
    answer = 1
    for i in range(1, k + 1):
        answer = answer * (n - k + i) // i
    return answer

### catalan

In [2]:
def catalan(n):
    if n < 0:
        raise ValueError('nonnegative n required')
    return binomial(2 * n, n) // (n + 1)

### count_onto_small

In [3]:
def count_onto_small(n, k):
    if min(n, k) < 0:
        raise ValueError('nonnegative sizes required')
    if k > n:
        return 0
    return sum(((-1) ** i * binomial(k, i) * (k - i) ** n for i in range(k + 1)))

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

n,k=4,3
show_table(['遗漏目标数i','选遗漏集合','剩余函数数','带符号贡献'],[(i,binomial(k,i),(k-i)**n,(-1)**i*binomial(k,i)*(k-i)**n) for i in range(k+1)])
print('满射数',count_onto_small(n,k))

遗漏目标数i,选遗漏集合,剩余函数数,带符号贡献
0,1,81,81
1,3,16,-48
2,3,1,3
3,1,0,0


满射数 36


## 7. 正确性、适用条件与复杂度

容斥中，一个恰好遗漏r个目标的函数被计数∑(-1)^i C(r,i)=(1−1)^r，因此只保留r=0的满射。Catalan=C(2n,n)−C(2n,n+1)=C(2n,n)/(n+1)。这些等式在整数域计算，不引入模逆元存在性的额外前提。

**代价：** 组合数O(min(k,n−k))次大整数乘除；Catalan同阶；小满射实现O(k²)级组合数算术上界，可预计算一行降为O(k)项。输出整数位数不可忽略；空集到空集的函数数为1。

### 展开理解

**容斥为什么恰好只留下满射？** 拿一个"恰好遗漏 r 个目标"的函数来审：它在 i=0 项被加了 1 次；在 i=1 项，只要被遗漏的目标在所选集合里就被减——C(r,1)=r 次；i=2 项被加回 C(r,2) 次……它对总和的净贡献是 ∑(−1)ⁱ C(r,i) = (1−1)ʳ。r > 0 时这个式子是 0（二项式定理展开 (1−1)ʳ），r = 0 时是 1。所以漏了目标的函数全部被加减抵消成 0，只有满射（r=0）每个恰好贡献 1——总和正好是满射个数。第三节表格里 i=3 那行为 0（因为 0⁴=0），你也能直观看到"全漏"的函数根本不存在。

拿一个具体函数走一遍抵消现场：n=4、k=3，考察那个"四个球全进 0 号盒"的函数（它遗漏了 1、2 号两个目标，r=2）。它在 i=0 项被数 1 次；在 i=1 项，"遗漏集合含 0 号盒"的选法有 C(2,1)=2 种（{0,1}、{0,2}）会数到它，被减 2 次；在 i=2 项，遗漏集合 {0,1,2} 数到它，被加回 1 次。净贡献 1 − 2 + 1 = 0，精确归零——这就是二项式 (1−1)² 展开的现场版。

**Catalan 公式为什么成立？** 反射法一句话版：把每对括号看成一步 +1/−1，合法串就是"任何前缀和非负、总和为 0"的路径。总路径数 C(2n,n)；非法路径第一次跌到 −1 时，把之后所有步镜像翻转，得到一条终点在另一侧的路径，这个翻转是可逆的一一配对，而那种路径共 C(2n,n+1) 条。相减就是合法数，代数上恰好等于 C(2n,n)/(n+1)。

**为什么敢用整除？** 两条整除都有整数为证：binomial 的每轮中间值是 C(n−k+i,i)（组合数，整数）；catalan 的 C(2n,n)−C(2n,n+1) 差值公式保证被 n+1 整除。所以全程精确，无浮点误差。

**复杂度（结合具体数字）：** binomial 做 O(min(k, n−k)) 次大整数乘除——n = 1000、k = 500 也就 500 轮，一瞬间；不过答案本身可能有约 k·log₁₀n 位（C(1000,500) 约 300 位），大整数乘除的代价随位数增长，“正确性与复杂度”部分 提醒"输出整数位数不可忽略"。catalan 同阶（多一次整除）。count_onto_small 是 O(k) 项求和、每项一次 binomial，整体 O(k²) 级；“正确性与复杂度”部分 提到可以预计算一行 Pascal 行把它降到 O(k) 项。小规模（比如 k ≤ 几十）完全够用。

## 8. 单元测试：每个用例在检查什么

1. `assert binomial(5,2)==10 and binomial(3,5)==0 and catalan(3)==5` —— **正常值 + 边界值**组合拳：C(5,2)=10 是标准抽查；C(3,5)=0 检查"k 超过 n 返回 0 而不是报错或乱算"；Cat(3)=5 对应 3 对括号的五种合法排列。
2. `for n in range(25): for k in range(n+1): assert binomial(n,k)==math.comb(n,k)` —— 拿标准库 `math.comb` 当裁判，n < 25 的整个下三角（含对角线 C(n,n)=1 和 C(n,0)=1）共 325 个值逐一对账。
3. Catalan 暴力对照（n = 0 到 5）：`product((1,-1),repeat=2*n)` 穷举长 2n 的全部 ±1 序列（n=5 时 1024 条），`total+=x` 逐步累加、`if total<0: good=False` 就是"前缀非负"判据，再要求 `total==0`（总数为零即左右括号各半）。数出来的合法条数和 `catalan(n)` 对照——这是把定义原样翻译成暴力，最有说服力的对账。
4. 满射暴力对照：`product(range(k),repeat=n)` 穷举全部 kⁿ 个函数（把每个函数写成值序列），`set(values)==set(range(k))` 判断"值域覆盖了所有 k 个目标"（无空盒），计数与 `count_onto_small(n,k)` 对照。注意 n=0、k=0 的角落：空集到空集恰好有 1 个函数（那个什么也不干的函数），“正确性与复杂度”部分 特意标注了这一点，暴力版天然给出 1，与公式版一致。

这组测试的总体策略是"公式版对照定义版"：binomial 对 `math.comb`，catalan 对 ±1 序列枚举，满射对函数穷举——三个公式全部被"笨办法"钉死在小规模上。

In [5]:
import math

from itertools import product

assert binomial(5, 2) == 10 and binomial(3, 5) == 0 and (catalan(3) == 5)

for n in range(25):
    for k in range(n + 1):
        assert binomial(n, k) == math.comb(n, k)

for n in range(6):
    valid = 0
    for seq in product((1, -1), repeat=2 * n):
        total = 0
        good = True
        for x in seq:
            total += x
            if total < 0:
                good = False
        valid += good and total == 0
    assert catalan(n) == valid
    for k in range(5):
        assert count_onto_small(n, k) == sum((set(values) == set(range(k)) for values in product(range(k), repeat=n)))

print('N129: 所有本章断言通过')

N129: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 推导不同BST数量递推。

**练习 2：** 比较抽球有序和无序计数。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（不同 BST 数量递推）：** 提示——n 个互不相同的节点构成二叉搜索树，关键观察是"选谁当根把问题切开"：固定根为值 i，左子树只能由比 i 小的 i−1 个节点构成、右子树由 n−i 个构成，两边独立，按乘法原理相乘，再对所有根选择求和，得 T(n) = ∑ T(i−1)·T(n−i)。手算前几项 T(0)=1, T(1)=1, T(2)=2, T(3)=5，你会发现和 Catalan 数一模一样——然后用 `catalan(n)` 验证这个猜想到 n=10 左右。

**练习 2（抽球有序 vs 无序）：** 提示——同一批球、同一次抽取，"抽出来排队"和"抽出来一堆"相差的恰好是每次抽取内部的排列因子。取 4 个球抽 2 个做手算：有序是 4×3=12，无序是 12÷2!=6；再算抽 3 个：24 vs 24÷6=4。用 `binomial` 验证无序一侧，自己写两三层循环枚举验证有序一侧；最后说清楚边界——抽的次数超过球数时有序、无序都该是 0。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
def binomial(n, k):
    if n < 0:
        raise ValueError('nonnegative n required')
    if k < 0 or k > n:
        return 0
    k = min(k, n - k)
    answer = 1
    for i in range(1, k + 1):
        answer = answer * (n - k + i) // i
    return answer

def catalan(n):
    if n < 0:
        raise ValueError('nonnegative n required')
    return binomial(2 * n, n) // (n + 1)

def count_onto_small(n, k):
    if min(n, k) < 0:
        raise ValueError('nonnegative sizes required')
    if k > n:
        return 0
    return sum(((-1) ** i * binomial(k, i) * (k - i) ** n for i in range(k + 1)))

# 示例与回归测试
import math

from itertools import product

assert binomial(5, 2) == 10 and binomial(3, 5) == 0 and (catalan(3) == 5)

for n in range(25):
    for k in range(n + 1):
        assert binomial(n, k) == math.comb(n, k)

for n in range(6):
    valid = 0
    for seq in product((1, -1), repeat=2 * n):
        total = 0
        good = True
        for x in seq:
            total += x
            if total < 0:
                good = False
        valid += good and total == 0
    assert catalan(n) == valid
    for k in range(5):
        assert count_onto_small(n, k) == sum((set(values) == set(range(k)) for values in product(range(k), repeat=n)))

print('N129: 所有本章断言通过')

N129: 所有本章断言通过


## 11. 代表题与迁移

- **118. Pascal's Triangle（杨辉三角）**：逐行生成杨辉三角——练的就是 C(n,k) = C(n−1,k−1) + C(n−1,k) 这条递推（本章"知识点"里的 Pascal 条目），用加法完全绕开除法。
- **96. Unique Binary Search Trees（不同的二叉搜索树）**：n 个节点能构成多少种 BST——正是本章练习 1 的原题，答案就是 Catalan 数，`catalan(n)` 直接可验。
- **62. Unique Paths（不同路径）**：m×n 网格只向右向下走——总步数固定、从中挑哪些步向右，答案是 C(m+n−2, m−1)，练"把走法问题翻译成组合数"的建模（notebook 标注 comparison，作为对照参考）。
- **920. Number of Music Playlists（播放列表数目）**：歌单要覆盖全部歌曲、允许部分条件下的重复——这是容斥/满射计数加上额外约束的进阶版（extension），考察把 `count_onto_small` 的思路迁移到带限制的场景。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。